**Author:** Dorys Trujillo  
**Project:** Legal Uncertainty Index (IIJ)   
**Data Source:** Ministry of Commerce, Industry and Tourism (MinCIT)  
**Period:** 2018–2025

### Text Vectorization (BoW and TF-IDF)

This notebook transforms the lemmatized corpus into numerical representations, including Bag of Words (BoW) and TF-IDF, to support subsequent co-occurrence analysis and index construction.

In [1]:
# Imports
from pathlib import Path
import pandas as pd
from collections import Counter

In [11]:
from pathlib import Path

# define project root (assumes execution inside notebooks/)
PROJECT_ROOT = Path().resolve().parents[0]

# input: lemmatized corpus
LEMMAS_ROOT = PROJECT_ROOT / "data_processed" / "corpus_clean" / "tokens_lemmas"

# output: vectorization results
VECTORIZATION_ROOT = PROJECT_ROOT / "data_processed" / "vectorization"
VECTORIZATION_ROOT.mkdir(parents=True, exist_ok=True)
# manifests: global tracking files
MANIFEST_ROOT = PROJECT_ROOT / "manifests"

# sanity checks
print("project_root:", PROJECT_ROOT)
print("lemmas_root exists:", LEMMAS_ROOT.exists(), LEMMAS_ROOT)
print("vectorization_root:", VECTORIZATION_ROOT)

# discover all documents
lemma_files = sorted(LEMMAS_ROOT.rglob("*.txt"))

#print(f"documents found: {len(lemma_files)}")

project_root: C:\Users\dtruj\Documentos\proyectos\legal-uncertainty-index
lemmas_root exists: True C:\Users\dtruj\Documentos\proyectos\legal-uncertainty-index\data_processed\corpus_clean\tokens_lemmas
vectorization_root: C:\Users\dtruj\Documentos\proyectos\legal-uncertainty-index\data_processed\vectorization


In [3]:
# read lemmatized documents
rows = []

for txt_path in lemma_files:
    rel_path = txt_path.relative_to(LEMMAS_ROOT)
    text = txt_path.read_text(encoding="utf-8", errors="ignore")
    tokens = [line.strip() for line in text.splitlines() if line.strip()]

    rows.append({
        "doc_id": txt_path.stem,
        "rel_path": str(rel_path),
        "n_tokens": len(tokens),
        "text": " ".join(tokens)
    })

df_docs = pd.DataFrame(rows)

print("documents loaded:", len(df_docs))
print("empty documents:", (df_docs["n_tokens"] == 0).sum())

display(df_docs[["doc_id", "rel_path", "n_tokens"]].head(10))

documents loaded: 333
empty documents: 0


,doc_id,rel_path,n_tokens
0,id_0085,Aplicación de derechos compensatorios\id_0085.txt,5127
1,id_0008,Arancel de Aduanas\id_0008.txt,52
2,id_0019,Arancel de Aduanas\id_0019.txt,86
3,id_0026,Arancel de Aduanas\id_0026.txt,112
4,id_0030,Arancel de Aduanas\id_0030.txt,64
5,id_0039,Arancel de Aduanas\id_0039.txt,107
6,id_0050,Arancel de Aduanas\id_0050.txt,71
7,id_0051,Arancel de Aduanas\id_0051.txt,150
8,id_0055,Arancel de Aduanas\id_0055.txt,106
9,id_0067,Arancel de Aduanas\id_0067.txt,163


### 1. Document-Term Matrix

The code constructs a Document-Term Matrix (DTM) from the lemmatized corpus using a Bag of Words approach. Each document is first represented as a sequence of cleaned tokens, which are then vectorized using CountVectorizer. The resulting matrix has documents as rows and unique terms as columns, where each cell contains the frequency of a given term within a specific document. This structured representation transforms the text corpus into a quantitative format, enabling further analysis such as vocabulary inspection, term frequency evaluation, and subsequent modeling steps.

In [4]:
from sklearn.feature_extraction.text import CountVectorizer

# build document-term matrix (bag of words)
vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(df_docs["text"])

# convert to dataframe
df_dtm = pd.DataFrame(
    X_bow.toarray(),
    index=df_docs["doc_id"],
    columns=vectorizer.get_feature_names_out()
)

# Transpose the result
#df_dtm_t = df_dtm.T
#display(df_dtm_t.iloc[:30, :30])

print("dtm shape:", df_dtm.shape)
display(df_dtm.iloc[:10, :10])

dtm shape: (333, 8656)


,abajo,abandona,abandonado,abandono,abarcar,abastecer,abastecido,abastecimiento,abierto,abiotico
doc_id,,,,,,,,,,
id_0085,0,0,0,0,2,0,0,0,0,0
id_0008,0,0,0,0,0,0,0,0,0,0
id_0019,0,0,0,0,0,0,0,0,0,0
id_0026,0,0,0,0,0,0,0,0,0,0
id_0030,0,0,0,0,0,0,0,0,0,0
id_0039,0,0,0,0,0,0,0,0,0,0
id_0050,0,0,0,0,0,0,0,0,0,0
id_0051,0,0,0,0,0,0,0,0,0,0
id_0055,0,0,0,0,0,0,0,0,0,0


In [5]:
# optional vocabulary check
#term_frequencies = df_dtm.sum(axis=0).sort_values(ascending=False)

#df_vocab = term_frequencies.reset_index()
#df_vocab.columns = ["term", "frequency"]

#print("vocabulary size:", len(df_vocab))
#display(df_vocab.head(30))

### 2. Term Frequency–Inverse Document Frequency

The TF-IDF (Term Frequency–Inverse Document Frequency) matrix is constructed to quantify the relative importance of terms across the corpus. Using the lemmatized documents as input, each term is weighted based on its frequency within a document and its inverse frequency across all documents, thereby reducing the influence of commonly occurring terms and emphasizing more informative ones. This representation provides a normalized and discriminative view of the textual data, facilitating subsequent analyses such as term relevance evaluation and feature selection for co-occurrence modeling.

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

# build TF-IDF matrix
tfidf_vectorizer = TfidfVectorizer()

X_tfidf = tfidf_vectorizer.fit_transform(df_docs["text"])

df_tfidf = pd.DataFrame(
    X_tfidf.toarray(),
    index=df_docs["doc_id"],
    columns=tfidf_vectorizer.get_feature_names_out()
)

print("tfidf shape:", df_tfidf.shape)
display(df_tfidf.iloc[:30, :30])

tfidf shape: (333, 8656)


,abajo,abandona,abandonado,abandono,abarcar,abastecer,abastecido,abastecimiento,abierto,abiotico,...,abordar,abreviado,abreviar,abreviatura,abril,abrir,absolutamente,absoluto,absolver,absorber
doc_id,,,,,,,,,,,,,,,,,,,,,
id_0085,0.0,0.0,0.0,0.0,0.009314,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.030915,0.0,0.004416,0.0,0.004979
id_0008,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000
id_0019,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000
id_0026,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000
id_0030,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000
id_0039,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000
id_0050,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000
id_0051,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000
id_0055,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000


In [7]:
# get top tf-idf terms by document
top_n = 10

rows = []

for doc_id in df_tfidf.index:
    top_terms = df_tfidf.loc[doc_id].sort_values(ascending=False).head(top_n)

    for term, score in top_terms.items():
        if score > 0:
            rows.append({
                "doc_id": doc_id,
                "term": term,
                "tfidf_score": score
            })

df_top_tfidf = pd.DataFrame(rows)

print("top tf-idf rows:", len(df_top_tfidf))
display(df_top_tfidf.head(40))

top tf-idf rows: 3330


,doc_id,term,tfidf_score
0,id_0085,subvencion,0.466044
1,id_0085,investigacion,0.279472
2,id_0085,dano,0.274154
3,id_0085,compensatorio,0.234018
4,id_0085,investigador,0.199164
5,id_0085,produccion,0.183993
6,id_0085,rama,0.155362
7,id_0085,importante,0.153747
8,id_0085,determinacion,0.126669
9,id_0085,producto,0.126653


The resulting matrices are stored to ensure reproducibility and to enable efficient downstream analysis, including co-occurrence modeling and index construction.

In [8]:
# output files
DTM_PATH = VECTORIZATION_ROOT / "dtm_bow.csv"
TFIDF_PATH = VECTORIZATION_ROOT / "dtm_tfidf.csv"
VOCAB_PATH = VECTORIZATION_ROOT / "vocabulary.csv"
TOP_TFIDF_PATH = VECTORIZATION_ROOT / "top_tfidf_terms.csv"

# save matrices
df_dtm.to_csv(DTM_PATH, encoding="utf-8")
df_tfidf.to_csv(TFIDF_PATH, encoding="utf-8")

# save vocabulary
term_frequencies = df_dtm.sum(axis=0).sort_values(ascending=False)

df_vocab = term_frequencies.reset_index()
df_vocab.columns = ["term", "frequency"]

df_vocab.to_csv(VOCAB_PATH, index=False, encoding="utf-8")

# save top tf-idf terms
df_top_tfidf.to_csv(TOP_TFIDF_PATH, index=False, encoding="utf-8")

print("dtm saved to:", DTM_PATH)
print("tf-idf saved to:", TFIDF_PATH)
print("vocabulary saved to:", VOCAB_PATH)
print("top tf-idf terms saved to:", TOP_TFIDF_PATH)

dtm saved to: C:\Users\dtruj\Documentos\proyectos\legal-uncertainty-index\data_processed\vectorization\dtm_bow.csv
tf-idf saved to: C:\Users\dtruj\Documentos\proyectos\legal-uncertainty-index\data_processed\vectorization\dtm_tfidf.csv
vocabulary saved to: C:\Users\dtruj\Documentos\proyectos\legal-uncertainty-index\data_processed\vectorization\vocabulary.csv
top tf-idf terms saved to: C:\Users\dtruj\Documentos\proyectos\legal-uncertainty-index\data_processed\vectorization\top_tfidf_terms.csv


In [14]:
# build manifest for vectorization stage

rows = []

for i, row in df_docs.iterrows():
    doc_id = row["doc_id"]

    rows.append({
        "doc_id": doc_id,
        "rel_path": row["rel_path"],
        "n_tokens": row["n_tokens"],
        "n_unique_terms_bow": int((df_dtm.loc[doc_id] > 0).sum()),
        "n_nonzero_tfidf": int((df_tfidf.loc[doc_id] > 0).sum()),
        "total_terms_bow": int(df_dtm.loc[doc_id].sum()),
        "status": "ok"
    })

df_manifest_vectorization = pd.DataFrame(rows)

# sort by number of tokens (descending)
df_manifest_vectorization = df_manifest_vectorization.sort_values(by="n_tokens", ascending=False)

# save in manifests folder
MANIFEST_VECTORIZATION_PATH = MANIFEST_ROOT / "manifest_vectorization.csv"
df_manifest_vectorization.to_csv(MANIFEST_VECTORIZATION_PATH, index=False, encoding="utf-8")

print("manifest saved to:", MANIFEST_VECTORIZATION_PATH)
display(df_manifest_vectorization.head(15))

manifest saved to: C:\Users\dtruj\Documentos\proyectos\legal-uncertainty-index\manifests\manifest_vectorization.csv


,doc_id,rel_path,n_tokens,n_unique_terms_bow,n_nonzero_tfidf,total_terms_bow,status
289,id_0104,Estructura superintendencia de sociedades\id_0...,7360,1307,1307,7360,ok
117,id_0113,Decreto 1074 de 2015\id_0113.txt,7295,1488,1488,7295,ok
104,id_0076,Decreto 1074 de 2015\id_0076.txt,7208,1396,1396,7208,ok
228,id_0099,Decreto 2147 de 2016\id_0099.txt,6543,1209,1209,6543,ok
132,id_0148,Decreto 1074 de 2015\id_0148.txt,5950,1394,1394,5950,ok
170,id_0250,Decreto 1074 de 2015\id_0250.txt,5290,1123,1123,5290,ok
326,id_0056,Sistemas especiales importExport\id_0056.txt,5203,973,973,5203,ok
327,id_0095,Sistemas especiales importExport\id_0095.txt,5165,1038,1038,5165,ok
95,id_0053,Decreto 1074 de 2015\id_0053.txt,5153,1319,1319,5153,ok
0,id_0085,Aplicación de derechos compensatorios\id_0085.txt,5127,1296,1296,5127,ok
